# ViFinQA — Issuer-Held-Out BGE-M3 Evaluation V1

Compare the trained BGE-M3 retriever with its untouched base model on `validation` and `test` issuers only. This notebook never reads benchmark questions, produces a submission, or promotes a retriever.

**Success criteria:** every source/model/curriculum gate passes; the output includes hash-bound Recall@K, MRR, and Top-1 wrong-entity/year/scope breakdown for both models.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

# Disable external experiment telemetry for the non-interactive Kaggle job.
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

REPO_DIR = Path('/kaggle/working/AI_guru_evaluation')
SOURCE_TREE_NAME = 'ai_guru_synthetic_retriever_source_v1'
SOURCE_MANIFEST_NAME = 'ai_guru_synthetic_retriever_source_v1.manifest.json'
source_dirs = sorted({
    path.parent for path in Path('/kaggle/input').rglob(SOURCE_MANIFEST_NAME)
    if (path.parent / SOURCE_TREE_NAME).is_dir()
})
if len(source_dirs) != 1:
    raise RuntimeError(f'Expected exactly one verified source snapshot; found {len(source_dirs)}.')
source_dir = source_dirs[0]
manifest = json.loads((source_dir / SOURCE_MANIFEST_NAME).read_text(encoding='utf-8'))
source_root = source_dir / SOURCE_TREE_NAME
platform_ignored_paths = {'pax_global_header'}
actual_files = {
    path.relative_to(source_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in source_root.rglob('*')
    if path.is_file() and path.relative_to(source_root).as_posix() not in platform_ignored_paths
}
actual_tree_sha = hashlib.sha256(json.dumps(actual_files, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
if (
    manifest.get('protocol') != 'kaggle_synthetic_retriever_source_v1'
    or actual_tree_sha != manifest.get('source_tree_sha256')
    or len(actual_files) != manifest.get('source_tree_file_count')
):
    raise ValueError('Synthetic source tree failed manifest/hash verification.')
if any(path == 'data/ViFinQA' or path.startswith('data/ViFinQA/') for path in actual_files):
    raise ValueError('Synthetic source snapshot must not contain benchmark data.')
if REPO_DIR.exists():
    raise RuntimeError(f'Refusing to merge source snapshot into existing path: {REPO_DIR}')
shutil.copytree(source_root, REPO_DIR)
evaluator = REPO_DIR / 'scripts' / 'evaluate_synthetic_retriever_v1.py'
if not evaluator.is_file():
    raise FileNotFoundError('Verified source snapshot is missing the held-out evaluator.')
print({'source_commit': manifest.get('git_commit'), 'source_tree_files': len(actual_files), 'evaluator': str(evaluator)})

In [ ]:
# Pin the same CUDA-compatible stack used for fine-tuning before importing model code.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'torch==2.12.1', 'torchvision==0.27.1',
    '--index-url', 'https://download.pytorch.org/whl/cu126',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'sentence-transformers==3.4.1', 'transformers==4.48.3',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, restart, then Run All.')
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gib < 14:
    raise RuntimeError(f'Issuer-held-out evaluation requires >=14 GiB VRAM; found {vram_gib:.1f} GiB.')
print({'gpu': torch.cuda.get_device_name(0), 'vram_gib': round(vram_gib, 2), 'torch': torch.__version__})

## Inputs and preflight gates

This kernel attaches: (1) the synthetic curriculum Dataset, (2) independent baseline `table_assets.jsonl`, (3) the verified source snapshot, and (4) output from the completed training kernel. It searches runtime paths rather than assuming Kaggle mount names.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
CURRICULUM_NAME = 'synthetic_finance_curriculum_v1.jsonl'
CURRICULUM_MANIFEST_NAME = 'synthetic_finance_curriculum_v1.manifest.json'

curriculum_dirs = sorted({
    path.parent for path in INPUT_ROOT.rglob(CURRICULUM_NAME)
    if (path.parent / CURRICULUM_MANIFEST_NAME).is_file()
})
if len(curriculum_dirs) != 1:
    raise RuntimeError(f'Expected exactly one curriculum input directory; found {len(curriculum_dirs)}.')
CURRICULUM_DIR = curriculum_dirs[0]
CURRICULUM = CURRICULUM_DIR / CURRICULUM_NAME
CURRICULUM_MANIFEST = CURRICULUM_DIR / CURRICULUM_MANIFEST_NAME

table_assets = sorted(INPUT_ROOT.rglob('table_assets.jsonl'))
if len(table_assets) != 1:
    raise RuntimeError(f'Expected exactly one independent table_assets.jsonl input; found {len(table_assets)}.')
TABLES = table_assets[0]

model_dirs = sorted({
    path.parent for path in INPUT_ROOT.rglob('training_metadata.json')
    if (path.parent / 'model.safetensors').is_file() and (path.parent / 'modules.json').is_file()
})
if len(model_dirs) != 1:
    raise RuntimeError(f'Expected exactly one trained model output directory; found {len(model_dirs)}.')
FINETUNED_MODEL = model_dirs[0]
training_metadata = json.loads((FINETUNED_MODEL / 'training_metadata.json').read_text(encoding='utf-8'))
if training_metadata.get('provenance') != 'synthetic_execution_verified':
    raise ValueError('Trained model metadata has an unexpected provenance.')
print({'curriculum': str(CURRICULUM), 'tables': str(TABLES), 'finetuned_model': str(FINETUNED_MODEL), 'training_examples': training_metadata.get('training_examples')})

## Evaluation

The base and fine-tuned models encode exactly the same full table corpus. Queries are only the issuer-held-out synthetic `validation` and `test` rows. The evaluator checks all source/curriculum hashes and replays every synthetic row before importing model libraries.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/bge_m3_issuer_heldout_evaluation_v1')
command = [
    sys.executable, str(evaluator),
    '--curriculum', str(CURRICULUM),
    '--manifest', str(CURRICULUM_MANIFEST),
    '--bundle-tables', str(TABLES),
    '--output-dir', str(OUTPUT_DIR),
    '--model', 'base=BAAI/bge-m3',
    '--model', f'finetuned={FINETUNED_MODEL}',
    '--splits', 'validation', 'test',
    '--ks', '1', '3', '5', '10', '20',
    '--passage-batch-size', '16', '--query-batch-size', '32',
    '--max-seq-length', '384', '--device', 'cuda:0',
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
result = json.loads((OUTPUT_DIR / 'evaluation_manifest.json').read_text(encoding='utf-8'))
if result.get('promotion_status') != 'offline_evaluation_complete_not_promoted':
    raise ValueError('Unexpected promotion status in evaluation artifact.')
summary = {
    label: {
        split: {
            'mrr': round(metrics['mrr'], 4),
            'recall@1': round(metrics['recall_at_k']['1'], 4),
            'recall@10': round(metrics['recall_at_k']['10'], 4),
            'top1_errors': metrics['top1_error_counts'],
        }
        for split, metrics in model_result['splits'].items()
    }
    for label, model_result in result['models'].items()
}
print(json.dumps({'summary': summary, 'status': result['promotion_status']}, ensure_ascii=False, indent=2))
print({'artifact_files': sorted(path.name for path in OUTPUT_DIR.iterdir())})

## Decision boundary

A completed evaluation produces comparison evidence only. Promote the fine-tuned retriever only after an explicit review confirms improved Recall@K **and** no unacceptable increase in Top-1 `wrong_scope` or `wrong_year`. Do not use this output as direct answer evidence or a contest submission.